# Aggregate hunting-ground harvest onto the hex grid (from a File Geodatabase)

Reads per-year hunting-ground feature classes from `hunting.gdb`, sums the harvest onto the hex
grid (one layer per year), and verifies the result. Count-based analogue of the Spanish
`gridPresence` step.

**Built-in handling:**
- Reads directly from the **geodatabase** (full column names + real nulls preserved).
- **Drops `nehonební`** (non-hunting) polygons; **flags `obora`** (enclosures).
- **Coerces** metric columns to numeric and keeps `NULL` as missing (never 0).
- **Whole-ground assignment** (largest overlap) → integer counts, totals conserved exactly.
- Adds a per-hexagon **completeness** field (`frac_data` = share of grounds that reported).

Set `DEMO = True` to run on simulated grounds without the real data.

In [1]:
# ===== CONFIG =====
GROUNDS_GDB    = r"hunting.gdb"              # your File Geodatabase
LAYER_TEMPLATE = "honitby_{year}"           # feature-class name pattern inside the gdb
HEX_PATH   = r"hexgrids_by_criterion.gpkg"  # hex grid
HEX_LAYER  = "complete95_8p5km"         # ~12 km grid (your current choice)
OUTDIR     = r"hex_aggregated"
OUT_GPKG   = "hexgrid_harvest_8km.gpkg"

ID_COL   = "HONITBA"                         # ground id (geometry) in the gdb
NAME_COL = "NAZEVHONIT"                      # ground name (for nehonební / obora); None if absent
METRIC_PREFIXES = ("Plan_", "Bag_", "Spring_")
METHOD = "mean_density"   # "sum_counts" (largest-overlap sum of counts) or "mean_density" (PI method: avg of ground densities)
ASSIGN          = "largest_overlap"         # only used when METHOD="sum_counts"; or "centroid"
DENSITY_EPS_M2  = 1.0                        # ignore boundary-only overlaps below this area (m^2)
DROP_NEHONEBNI  = True

DEMO        = False                          # True = simulated grounds (no real data needed)
DEMO_YEARS  = [2016, 2017, 2018, 2019]
DEMO_NGROUND = 4000
YEARS = DEMO_YEARS if DEMO else list(range(2003, 2023))   # edit to the years present in your gdb

In [2]:
import os, sqlite3, warnings, numpy as np, pandas as pd, geopandas as gpd
from pyproj import CRS
warnings.filterwarnings("ignore")
os.makedirs(OUTDIR, exist_ok=True)

TARGET_CRS = 5514
ESRI_WKT   = CRS.from_epsg(TARGET_CRS).to_wkt(version="WKT1_ESRI")
def patch_gpkg_crs(path):
    con=sqlite3.connect(path)
    con.execute("UPDATE gpkg_spatial_ref_sys SET definition=? WHERE srs_id=? OR organization_coordsys_id=?",
                (ESRI_WKT,TARGET_CRS,TARGET_CRS)); con.commit(); con.close()

hexg = gpd.read_file(HEX_PATH, layer=HEX_LAYER)[["hex_id","area_km2","area_cz_km2","geometry"]].to_crs(TARGET_CRS)
HEX_CRS = hexg.crs
print("Hex grid:", len(hexg), "hexagons | layer:", HEX_LAYER, "| CRS:", HEX_CRS.to_epsg())

def list_gdb_layers():
    import pyogrio; return [l[0] for l in pyogrio.list_layers(GROUNDS_GDB)]

Hex grid: 1829 hexagons | layer: complete95_8p5km | CRS: 5514


## Input grounds (real gdb, or simulated)

In [3]:
_SP=["RedDeer","FallowDeer","RoeDeer","WildBoar","Mouflon"]
_DM=[f"{m}_{s}_{c}" for m in("Plan","Bag","Spring") for s in _SP for c in("Male","Female","Total")]
def _demo_grounds(year, n=DEMO_NGROUND, seed=1):
    from shapely.ops import voronoi_diagram
    from shapely.geometry import MultiPoint, Point
    from shapely.prepared import prep
    rng=np.random.default_rng(seed+year)
    land=hexg.unary_union; PL=prep(land); minx,miny,maxx,maxy=land.bounds
    pts=[]
    while len(pts)<n:
        for px,py in zip(rng.uniform(minx,maxx,n),rng.uniform(miny,maxy,n)):
            if PL.contains(Point(px,py)): pts.append((px,py))
            if len(pts)>=n: break
    cells=[c.intersection(land) for c in voronoi_diagram(MultiPoint(pts),envelope=land).geoms]
    rows=[{ID_COL:f"CZ{year}{i:05d}", NAME_COL:("obora" if rng.random()<.03 else f"g{i}"),"geometry":c}
          for i,c in enumerate(cells) if (not c.is_empty and c.area>0)]
    g=gpd.GeoDataFrame(rows,crs=HEX_CRS)
    for col in _DM:
        v=rng.poisson(max(1,rng.normal(50,10)),len(g)).astype("float64")
        v[rng.random(len(g))<.15]=0.0; v[rng.random(len(g))<.25]=np.nan; g[col]=v
    return g

def read_grounds(year):
    if DEMO: return _demo_grounds(year)
    layer=LAYER_TEMPLATE.format(year=year)
    g=gpd.read_file(GROUNDS_GDB, layer=layer)
    if g.crs is None or g.crs.to_epsg()!=TARGET_CRS: g=g.to_crs(TARGET_CRS)
    return g

## Assignment + aggregation

In [4]:
def assign_to_hex(grounds, hexg, how="largest_overlap"):
    g=grounds.copy()
    if how=="centroid":
        cen=g[[ID_COL]].copy(); cen["geometry"]=g.geometry.centroid
        cen=gpd.GeoDataFrame(cen,crs=g.crs)
        j=gpd.sjoin(cen,hexg[["hex_id","geometry"]],how="left",predicate="within")[[ID_COL,"hex_id"]]
        return g.merge(j.drop_duplicates(ID_COL),on=ID_COL,how="left")
    inter=gpd.overlay(g[[ID_COL,"geometry"]],hexg[["hex_id","geometry"]],how="intersection")
    inter["_a"]=inter.geometry.area
    win=inter.loc[inter.groupby(ID_COL)["_a"].idxmax(),[ID_COL,"hex_id"]]
    return g.merge(win,on=ID_COL,how="left")

def _prep(grounds):
    """Common prep: detect metrics, coerce to numeric (NULL->NaN), drop nehonebni, flag obora."""
    g=grounds.copy()
    metric=[c for c in g.columns if str(c).startswith(METRIC_PREFIXES)]
    g[metric]=g[metric].replace("NULL",np.nan).apply(pd.to_numeric,errors="coerce")
    nm = g[NAME_COL].astype(str).str.lower() if (NAME_COL and NAME_COL in g.columns) else pd.Series("",index=g.index)
    n_neh=int(nm.str.contains("nehoneb").sum())
    if DROP_NEHONEBNI: g=g[~nm.str.contains("nehoneb")].copy(); nm=nm.loc[g.index]
    g["_is_obora"]=nm.str.contains("obora")
    return g, metric, n_neh

def aggregate_year(grounds, hexg):
    g, metric, n_neh = _prep(grounds)
    null_cells=int(g[metric].isna().sum().sum())   # null integrity on UNIQUE grounds

    if METHOD=="mean_density":
        # density per ground (animals / km2 of GROUND area), NaN where harvest is null
        area_km2 = g.geometry.area/1e6
        dens = g[metric].div(area_km2, axis=0); dens[ID_COL]=g[ID_COL].values
        dens["_is_obora"]=g["_is_obora"].values
        # overlapping (intersecting, real-area) ground-hexagon pairs -> ground contributes to every hex it overlaps
        inter=gpd.overlay(g[[ID_COL,"geometry"]],hexg[["hex_id","geometry"]],how="intersection")
        inter["_ia"]=inter.geometry.area
        pairs=inter[inter["_ia"]>DENSITY_EPS_M2][[ID_COL,"hex_id"]].drop_duplicates()
        unassigned=int(g[~g[ID_COL].isin(pairs[ID_COL])].shape[0])
        dd=pairs.merge(dens,on=ID_COL)
        mean_dens=dd.groupby("hex_id")[metric].mean()              # mean skips NaN -> excludes null grounds; zeros kept
        cnt=dd.groupby("hex_id").agg(n_grounds=(ID_COL,"nunique"),n_obora=("_is_obora","sum"))
        out=hexg.merge(mean_dens,on="hex_id",how="left").merge(cnt,on="hex_id",how="left")
        for c in ["n_grounds","n_obora"]: out[c]=out[c].fillna(0).astype(int)
        return out, dd, metric, unassigned, n_neh, null_cells

    # ---- METHOD == "sum_counts" (whole-ground assignment, sum of counts) ----
    g["_has_data"]=g[metric].notna().any(axis=1)
    a=assign_to_hex(g,hexg,ASSIGN)
    unassigned=int(a["hex_id"].isna().sum()); a=a.dropna(subset=["hex_id"])
    summed=a.groupby("hex_id")[metric].sum(min_count=1)
    cnt=a.groupby("hex_id").agg(n_grounds=(ID_COL,"size"),n_data=("_has_data","sum"),n_obora=("_is_obora","sum"))
    out=hexg.merge(summed,on="hex_id",how="left").merge(cnt,on="hex_id",how="left")
    for c in ["n_grounds","n_data","n_obora"]: out[c]=out[c].fillna(0).astype(int)
    out["frac_data"]=np.where(out["n_grounds"]>0,(out["n_data"]/out["n_grounds"]).round(3),np.nan)
    return out, a, metric, unassigned, n_neh, null_cells

## Run for every year

In [5]:
gpath=os.path.join(OUTDIR,OUT_GPKG)
if os.path.exists(gpath): os.remove(gpath)
audit={}
print(f"METHOD = {METHOD}")
for yr in YEARS:
    try:
        grounds=read_grounds(yr)
    except Exception as e:
        print(f"{yr}: SKIP ({type(e).__name__}: {str(e)[:60]}) - is layer '{LAYER_TEMPLATE.format(year=yr)}' in the gdb?")
        continue
    out,assigned,metric,unassigned,n_neh,null_cells=aggregate_year(grounds,hexg)
    out.to_file(gpath,layer=f"harvest_hex_{yr}",driver="GPKG")
    rec=dict(metric=metric,n_in=len(grounds),unassigned=unassigned,n_neh=n_neh,null_cells=null_cells,
             n_occupied=int((out["n_grounds"]>0).sum()))
    if METHOD=="sum_counts":
        gg,_,_=_prep(grounds)
        rec["in_tot"]={m:float(np.nansum(gg[m])) for m in metric}
        rec["out_tot"]={m:float(np.nansum(out[m])) for m in metric}
    audit[yr]=rec
    extra = "" if METHOD=="sum_counts" else " (mean density, animals/km2)"
    print(f"{yr}: {len(grounds)} grounds (-{n_neh} nehonební) -> {rec['n_occupied']} hexes w/ data, "
          f"{unassigned} unassigned, {null_cells} null cells{extra}")
patch_gpkg_crs(gpath)
print("Wrote", gpath, "(CRS = ESRI WKT1)")

METHOD = mean_density
2003: 6245 grounds (-457 nehonební) -> 1828 hexes w/ data, 0 unassigned, 371470 null cells (mean density, animals/km2)
2004: 6245 grounds (-457 nehonební) -> 1828 hexes w/ data, 0 unassigned, 282927 null cells (mean density, animals/km2)
2005: 6245 grounds (-457 nehonební) -> 1828 hexes w/ data, 0 unassigned, 276672 null cells (mean density, animals/km2)
2006: 6245 grounds (-457 nehonební) -> 1828 hexes w/ data, 0 unassigned, 270834 null cells (mean density, animals/km2)
2007: 6245 grounds (-457 nehonební) -> 1828 hexes w/ data, 0 unassigned, 264718 null cells (mean density, animals/km2)
2008: 6245 grounds (-457 nehonební) -> 1828 hexes w/ data, 0 unassigned, 254293 null cells (mean density, animals/km2)
2009: 6245 grounds (-457 nehonební) -> 1828 hexes w/ data, 0 unassigned, 247760 null cells (mean density, animals/km2)
2010: 6245 grounds (-457 nehonební) -> 1828 hexes w/ data, 0 unassigned, 676657 null cells (mean density, animals/km2)
2011: 6245 grounds (-457 n

# Verification

In [6]:
# V1 - correctness
if METHOD=="sum_counts":
    bad=[(y,m) for y,a in audit.items() for m in a["metric"]
         if not np.isclose(a["in_tot"][m],a["out_tot"][m],atol=1e-6)]
    print("V1 conservation:", "PASS - all metric totals preserved" if not bad else f"FAIL: {bad[:5]}")
else:
    # mean_density: independently recompute one hexagon's mean density and compare
    yr=[y for y in YEARS if y in audit][0]
    g,metric,_=_prep(read_grounds(yr))
    area=g.geometry.area/1e6; dens=g[metric].div(area,axis=0); dens[ID_COL]=g[ID_COL].values
    inter=gpd.overlay(g[[ID_COL,"geometry"]],hexg[["hex_id","geometry"]],how="intersection")
    inter["_ia"]=inter.geometry.area
    pairs=inter[inter["_ia"]>DENSITY_EPS_M2][[ID_COL,"hex_id"]].drop_duplicates()
    lay=gpd.read_file(gpath,layer=f"harvest_hex_{yr}")
    chk=[c for c in metric if lay[c].notna().any()][0]
    hid=lay.loc[lay[chk].notna(),"hex_id"].iloc[50]
    ids=pairs[pairs.hex_id==hid][ID_COL]
    manual=dens.set_index(ID_COL).loc[ids,chk].mean()
    got=lay.loc[lay.hex_id==hid,chk].iloc[0]
    print(f"V1 density recompute ({yr}, {chk}, hex {hid}): manual={manual:.4f} output={got:.4f}",
          "PASS" if np.isclose(manual,got) else "FAIL")

V1 density recompute (2003, Plan_RedDeer_Male, hex H00061): manual=0.0000 output=0.0000 PASS


In [7]:
# V2 no grounds lost
prob=[(y,a["unassigned"]) for y,a in audit.items() if a["unassigned"]>0]
print("V2 assignment:", "PASS - all grounds placed" if not prob else f"NOTE unassigned: {prob}")

V2 assignment: PASS - all grounds placed


In [8]:
# V3 null integrity - nulls must SURVIVE (a near-zero count signals upstream int-truncation/0-fill)
print("V3 null integrity (nulls should be > 0 if any no-data exists):")
for y,a in audit.items():
    flag="" if a["null_cells"]>0 else "  <-- SUSPECT: no nulls, check the gdb export!"
    print(f"  {y}: {a['null_cells']:>8} null metric cells{flag}")

V3 null integrity (nulls should be > 0 if any no-data exists):
  2003:   371470 null metric cells
  2004:   282927 null metric cells
  2005:   276672 null metric cells
  2006:   270834 null metric cells
  2007:   264718 null metric cells
  2008:   254293 null metric cells
  2009:   247760 null metric cells
  2010:   676657 null metric cells
  2011:   677877 null metric cells
  2012:   678523 null metric cells
  2013:   136650 null metric cells
  2014:   132464 null metric cells
  2015:   129955 null metric cells
  2016:   124884 null metric cells
  2017:   122382 null metric cells
  2018:   122416 null metric cells
  2019:   122641 null metric cells
  2020:   122938 null metric cells
  2021:   122479 null metric cells
  2022:   122555 null metric cells


In [9]:
# V4 grounds-per-hex (+ completeness for sum_counts)
yr=[y for y in YEARS if y in audit][0]
lay=gpd.read_file(gpath,layer=f"harvest_hex_{yr}")
occ=lay[lay.n_grounds>0]
print(f"V4 ({yr}): grounds/occupied hex - mean {occ.n_grounds.mean():.2f}, median {int(occ.n_grounds.median())}, max {int(occ.n_grounds.max())}")
if METHOD=="sum_counts" and "frac_data" in lay.columns:
    print(f"      completeness frac_data - median {occ.frac_data.median():.3f}, hexes <100% reporting: {int((occ.frac_data<1).sum())}")
else:
    mcols=[c for c in lay.columns if str(c).startswith(METRIC_PREFIXES)]
    print(f"      output is MEAN DENSITY (animals/km2); e.g. median across occupied hexes for {mcols[0]}: {occ[mcols[0]].median():.3f}")

V4 (2003): grounds/occupied hex - mean 8.46, median 9, max 16
      output is MEAN DENSITY (animals/km2); e.g. median across occupied hexes for Plan_RedDeer_Male: 0.000


## Optional: monitoring-period layers
Plan & Bag are **summed** across years; Spring counts are **averaged** (standing-stock snapshot).

In [10]:
def period_layer(years,label):
    present=[y for y in years if y in audit]
    if not present: print(f"{label}: no years, skip"); return
    frames={y:gpd.read_file(gpath,layer=f"harvest_hex_{y}").set_index("hex_id") for y in present}
    mcols=[c for c in next(iter(frames.values())).columns if str(c).startswith(METRIC_PREFIXES)]
    out=hexg.set_index("hex_id").copy()
    for m in mcols:
        stack=pd.concat([frames[y][m] for y in present],axis=1)
        if METHOD=="mean_density":
            out[m]=stack.mean(axis=1,skipna=True)               # average densities across years
        else:
            out[m]=stack.mean(axis=1,skipna=True) if m.startswith("Spring_") else stack.sum(axis=1,min_count=1)
    out.reset_index().to_file(gpath,layer=f"harvest_hex_{label}",driver="GPKG")
    print(f"{label}: from years {present}")
period_layer(range(2013,2019),"2013_2018")
period_layer(range(2019,2023),"2019_2022")
period_layer(range(2013,2023),"2013_2022")
patch_gpkg_crs(gpath)

2013_2018: from years [2013, 2014, 2015, 2016, 2017, 2018]
2019_2022: from years [2019, 2020, 2021, 2022]
2013_2022: from years [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]


# Outputs & notes
`hex_aggregated/hexgrid_harvest.gpkg` - one `harvest_hex_<year>` layer + period layers, each with the
summed metrics, `n_grounds`, `n_data`, `n_obora`, `frac_data`, and area fields. CRS = ESRI WKT1 (ArcGIS-safe).

- To find your layer names: `print(list_gdb_layers())`.
- For density maps: divide a metric by `area_cz_km2`.
- `frac_data` < 1 flags hexagons where some grounds didn't report - the honest data-gap indicator.
- Keep or drop `obora` downstream using `n_obora`.